In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import gymnasium as gym
import numpy as np
from collections import defaultdict
from RL_Algorithms import RL_Algorithm
from Updaters import *
from Strategies import *
from State_Representations import *


In [3]:
env = gym.make("Pendulum-v1")

obs_cardinality = (12, 12, 12)  # cos(theta), sin(theta), theta_dot
obs_space_high = np.array([1.0, 1.0, 8.0])
obs_space_low = np.array([-1.0, -1.0, -8.0])

n_actions = 9
action_space = np.linspace(-2, 2, n_actions)

episodes = 100

In [4]:

class classic_Q_Learning(RL_Algorithm):
    def __init__(self, ID, alpha : float, gamma : float, eps_decay : float, eps_min : float):
        super().__init__()
        self.strategy = Epsilon_Decay(epsilon = 1.0, epsilon_decay = eps_decay, epsilon_min = eps_min)
        self.updater = Q_Learning(alpha=alpha, gamma=gamma)

class classic_SARSA(RL_Algorithm):
    def __init__(self, ID, alpha : float, gamma : float, eps_decay : float, eps_min : float):
        super().__init__()
        self.strategy = Epsilon_Decay(epsilon = 1.0, epsilon_decay = eps_decay, epsilon_min = eps_min)
        self.updater = SARSA(alpha=alpha, gamma=gamma)

In [5]:
gammas = [0.99, 0.9]
alphas = [0.1, 0.01]
eps_decays = [0.9995, 0.9999]
eps_mins = [0.05]

algoritms = {}

In [6]:
i = 0
for gamma in gammas:
    for alpha in alphas:
        for eps_decay in eps_decays:
            for eps_min in eps_mins:
                q = classic_Q_Learning(ID=i, alpha=alpha, gamma=gamma, eps_decay=eps_decay, eps_min=eps_min)
                algoritms[i] = q
                i += 1
                s = classic_SARSA(ID=i, alpha=alpha, gamma=gamma, eps_decay=eps_decay, eps_min=eps_min)
                algoritms[i] = s
                i += 1

In [7]:
algorithm_rewards = defaultdict(list)

alg : RL_Algorithm
for ix, (id, alg) in enumerate(algoritms.items()):
    q_table = Q_Table(obs_cardinality, obs_space_low, obs_space_high, (n_actions, ))
    alg.Set_State_Representation(q_table)

    for episode in range(episodes):
        obs, _ = env.reset()
        state = q_table.Observation_To_State(obs)

        total_reward = 0
        done = False

        while not done:

            action_index, next_state, reward, terminated, truncated = alg.Step(action_space, state, env)

            state = next_state
            total_reward += reward

            done = terminated or truncated

        alg.Episode_Ended()
        algorithm_rewards[id].append(total_reward)

        if (episode + 1) % 5000 == 0:
            print(f"Epizod {episode+1}, total reward: {total_reward:.2f}, epsilon: {alg.Get_Strategy().Get_Epsilon():.2f}")


    print(f"Trening zakończony! Total reward: {total_reward:.2f}")

Trening zakończony! Total reward: -1067.05
Trening zakończony! Total reward: -1737.45
Trening zakończony! Total reward: -874.58
Trening zakończony! Total reward: -1620.68
Trening zakończony! Total reward: -1311.43
Trening zakończony! Total reward: -1425.12
Trening zakończony! Total reward: -838.92
Trening zakończony! Total reward: -865.16
Trening zakończony! Total reward: -1435.63
Trening zakończony! Total reward: -1518.98
Trening zakończony! Total reward: -1019.76
Trening zakończony! Total reward: -984.29
Trening zakończony! Total reward: -1545.98
Trening zakończony! Total reward: -1779.25
Trening zakończony! Total reward: -1129.12
Trening zakończony! Total reward: -1656.62


In [ ]:
final_rewards : dict[int, float] = {}
for ix, (id, rewards) in enumerate(algorithm_rewards.items()):
    final_rewards[id] = rewards[-1].item()

final_rewards = dict(sorted(final_rewards.items(), key=lambda item: -item[1]))

print(final_rewards)
print(algoritms[(list(final_rewards.keys())[0])])

{6: -838.9198533589898, 7: -865.1617424103565, 2: -874.5820857224297, 11: -984.2894989795525, 10: -1019.763004236239, 0: -1067.0533214124794, 14: -1129.1172914096169, 4: -1311.4273373135925, 5: -1425.124439419655, 8: -1435.634917942078, 9: -1518.9805995376955, 12: -1545.9789486313111, 3: -1620.677994771462, 15: -1656.6188106844595, 1: -1737.4456204021708, 13: -1779.2507162051074}
Q-Learning: gamma=0.99, alpha=0.01 | Epsilon decay: decay=0.9999, eps_min=0.05


In [9]:

env = gym.make("Pendulum-v1", render_mode = "human")

obs, _ = env.reset()
state =  list(algoritms.values())[-1].Get_State_Representation().Observation_To_State(obs)
strategy = Greedy()
eval_rew = 0

for _ in range(200):
    action_idx : int = strategy.Get_Action_Index(q_table, state)
    action : np.array = np.array(action_space[action_idx])

    obs, rew, terminated, truncated, _ = env.step([action.item()])
    eval_rew += rew
    state = q_table.Observation_To_State(obs)

print(f"Eval rew: {eval_rew}")
env.close()

Eval rew: -878.1536150312287
